## Evaluation pipeline for the microlane experiment

In [7]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [8]:
# Imports of the Core Packages
import json, sys, time, pytz
import os, yaml,random 
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [9]:
# Import custom libraries located at different folder location + configs
from microlane.utils.metrics import *
from microlane.datasets.tusimple import TuSimple
from microlane.models.lanenet2.model import LaneNet2
from microlane.schema.output import ModelPrediction
from microlane.schema.sample import Sample
from microlane.utils.load_image import load_image_from_sample, parse_prediction
from microlane.utils.experiment import ExperimentEvaluate

In [10]:
# First Load the Configuation file
with open("configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Pre Processing Part

In [11]:
# First initialise the dataset
# Then load the dataset
dataset = TuSimple(
    
        folder_path=config['data']['datasets']['tusimple']['path'],
        
        annotation_file_path=config['data']['datasets']['tusimple']['annotation_file']
    )

data = dataset.load(number=500)

In [12]:
# So, basically now we will import the model
# model = LaneNet2() type and what we will do is, run 
# Run model.inference(formatted_dataset)
# Ensure that Docker Engine Is Running in background

model = LaneNet2(
    
    container_folder=config['models']['lanenet2']['container_folder'],
    
    image_name=config['models']['lanenet2']['image_name']
    
)

Initializing container on port  8000
/home/suyog/desktop/projects/microlane/microlane/models/lanenet2/lanenet2
Image 'lanenet2_image:latest' already exists, skipping build.
Starting new container from 'lanenet2_image:latest' on port 8000...
Container started: b7c2eeb60a8c
Waiting for container to be ready on port 8000...
Container is ready.


In [13]:
experiment = ExperimentEvaluate(
    
    experiment_name="BATCH pipeline testing with no augmentation "
    
)

### Models and Datasets Loaded, Now Processing Part

In [14]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

item = data[0]
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

Total items: 500

Image Path   : /home/suyog/assets/datasets/TuSimple/TUSimple/test_set/clips/0530/1492626760788443246_0/20.jpg
h_samples    : [160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 500, 510, 520, 530, 540, 550, 560, 570, 580, 590, 600, 610, 620, 630, 640, 650, 660, 670, 680, 690, 700, 710]
lanes        : [[-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, 648, 636, 626, 615, 605, 595, 585, 575, 565, 554, 545, 536, 526, 517, 508, 498, 489, 480, 470, 461, 452, 442, 433, 424, 414, 405, 396, 386, 377, 368, 359, 349, 340, 331, 321, 312, 303, 293, 284, 275, 265, 256, 247, 237, 228, 219], [-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, 681, 692, 704, 716, 728, 741, 754, 768, 781, 794, 807, 820, 834, 847, 860, 873, 886, 900, 913, 926, 939, 952, 966, 979, 992, 1005, 1018, 1032, 1045, 1058, 1071, 1084, 1098, 1111, 1124, 1137, 1150, 1164, 1177, 1190, 1203, 1216, 1230, 1243, 1256, 12

In [15]:
blur_range = tuple(config['data']['augmentation']['blur'])
rotation_range = tuple(config['data']['augmentation']['blur'])
zoom_range = tuple(config['data']['augmentation']['zoom'])
brightness_range =tuple(config['data']['augmentation']['brightness'])

### Looping through all the testing examples

In [ ]:
for testing_example in tqdm(data):
    
    
    ## Possible Data Augmentation that we could do here

    # item.blur       = round(random.uniform(blur_range[0],       blur_range[1] - 0.01), 2)
    # item.rotation   = round(random.uniform(rotation_range[0],   rotation_range[1] - 0.01), 2)
    # item.zoom       = round(random.uniform(zoom_range[0],       zoom_range[1] - 0.01), 2)
    # item.brightness = round(random.uniform(brightness_range[0], brightness_range[1] - 0.01), 2)

    loaded_image = load_image_from_sample(testing_example)
    
    response = model.predict(loaded_image)
    
    modeloutput = parse_prediction(response)
    
    experiment.store_prediction(modeloutput)
    
    experiment.visualize_prediction(modeloutput)

  0%|          | 0/500 [00:00<?, ?it/s]

[0] RAM: 468.9 MB | Top alloc: /home/suyog/desktop/projects/microlane/venv/lib/python3.12/site-packages/matplotlib/image.py:670: size=5400 KiB, count=4, average=1350 KiB
[10] RAM: 729.0 MB | Top alloc: /home/suyog/desktop/projects/microlane/venv/lib/python3.12/site-packages/matplotlib/image.py:670: size=58.0 MiB, count=44, average=1350 KiB
[20] RAM: 986.2 MB | Top alloc: /home/suyog/desktop/projects/microlane/venv/lib/python3.12/site-packages/matplotlib/image.py:670: size=111 MiB, count=84, average=1350 KiB


KeyboardInterrupt: 

In [ ]:
iters = [x[0] for x in memory_log]
rams  = [x[1] for x in memory_log]

plt.figure(figsize=(10, 4))
plt.plot(iters, rams)
plt.xlabel("Iteration")
plt.ylabel("RAM (MB)")
plt.title("Memory usage over inferences")
plt.grid(True)
plt.tight_layout()
plt.show()

# Print top allocators
print("\nTop memory lines:")
for entry in memory_log[-5:]:
    print(f"  iter {entry[0]:>3}: {entry[2]}")